# Reimplementation of GAT using Message Passing
According to [Inductive Representation Learning on Large Graphs](https://arxiv.org/abs/1706.02216) by William L. Hamilton, Rex Ying, Jure Leskovec

In [33]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, softmax

### A Brief Review on GAT's Archtecture
![GAT](./photos/GAT.jpg)

## 1. Linear Transformation

Each node's features are linearly transformed using a shared weight matrix:

$$
\mathbf{h}'_i = \mathbf{W} \mathbf{h}_i
$$

**Equation 1**: Linear transformation of input features.

## 2. Self-Loops

Self-loops are added to the adjacency matrix to include the node itself in its neighborhood.

**Reference**: "We perform multi-head self-attention on the nodes ... after adding self-loops to the input graph" (Section 3.3).

## 3. Attention Mechanism

The attention mechanism is a single-layer feedforward neural network:

$$
e_{ij} = \text{LeakyReLU}(\mathbf{a}^T [\mathbf{h}'_i || \mathbf{h}'_j])
$$

**Equation 2**: Computing unnormalized attention coefficients.

## 4. Softmax Normalization

Normalize the attention coefficients across all neighbors:

$$
\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k \in \mathcal{N}_i} \exp(e_{ik})}
$$

**Equation 3**: Normalized attention coefficients.

## 5. Dropout on Attention Coefficients

Apply dropout to the normalized attention coefficients for regularization.

## 6. Weighted Sum of Neighbors' Features

Compute a linear combination of the neighbors' features weighted by the attention coefficients:

$$
\mathbf{h}'_i = \sigma \left( \sum_{j \in \mathcal{N}_i} \alpha_{ij} \mathbf{W} \mathbf{h}_j \right)
$$

**Equation 4**: Aggregated node features using attention coefficients.

## 7. Multi-Head Attention

Concatenate or average the outputs of multiple attention heads:

$$
\mathbf{h}'_i = \Bigg\|_{k=1}^K \sigma \left( \sum_{j \in \mathcal{N}_i} \alpha_{ij}^k \mathbf{W}^k \mathbf{h}_j \right)
$$

**Equation 5**: Concatenation of multi-head attention outputs.

### A Brief Explanation of Code
1) __MessagePassing(aggr='add')__: Initializes the GAT layer with the "add" aggregation method.
2) __self.lin = torch.nn.Linear()__: Defines a linear layer to apply a shared linear transformation to the input features. h' = W h<sub>i</sub>
3) __self.att = torch.nn.Parameter()__: Defines the attention mechanism, a learnable parameter.
4) __edge_index, _ = add_self_loops()__: Adds self-loops to the adjacency matrix to include the node itself in its neighborhood.
5) __x = self.lin(x)__: Applies the linear transformation to the input features.
6) 
    - __x_j = torch.cat([x_i, x_j], dim=-1)__: Concatenates features of source and target nodes.
    - __alpha = (x_j * self.att).sum(dim=-1)__: Computes the attention coefficients.
    - __alpha = F.leaky_relu():__ Applies Leaky ReLU activation.
    - __alpha = softmax()__: Normalizes the attention coefficients using softmax.
7) __alpha = F.dropout()__: Applies dropout to the attention coefficients for regularization.
8) __return x_j * alpha.view(-1, self.heads, 1)__: Computes the weighted sum of the neighbors' features.

In [34]:
class GATConv(MessagePassing):
    def __init__(self, in_channels, out_channels, heads=1, concat=True, negative_slope=0.2, dropout=0):
        super(GATConv, self).__init__(aggr='add')  # "Add" aggregation (Step 1)
        self.heads = heads
        self.out_channels = out_channels
        self.concat = concat
        self.negative_slope = negative_slope
        self.dropout = dropout

        # Linear layer for shared weight transformation (Step 2)
        self.lin = torch.nn.Linear(in_channels, heads * out_channels, bias=False)
        
        # Attention mechanism (single-layer feedforward neural network) (Step 3)
        self.att = torch.nn.Parameter(torch.Tensor(1, heads, 2 * out_channels))
        
        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.xavier_uniform_(self.lin.weight)
        torch.nn.init.xavier_uniform_(self.att)

    def forward(self, x, edge_index):
        # Add self-loops to the adjacency matrix (Step 4)
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        
        # Linear transformation (Step 5)
        x = self.lin(x)
        x = x.view(-1, self.heads, self.out_channels)
        
        return self.propagate(edge_index, x=x)

    def message(self, x_i, x_j, edge_index_i, size_i):
        # Compute attention coefficients (Step 6)
        x_cat = torch.cat([x_i, x_j], dim=-1)
        alpha = (x_cat * self.att).sum(dim=-1)
        alpha = F.leaky_relu(alpha, self.negative_slope)
        alpha = softmax(alpha, edge_index_i, num_nodes=size_i)

        # Sample attention coefficients with dropout (Step 7)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)
        
        # Weighted sum of neighbors' features (Step 8)
        return x_j * alpha.view(-1, self.heads, 1)
    
    def update(self, aggr_out):
        if self.concat:
            # Concat heads' outputs (Step 9)
            return aggr_out.view(-1, self.heads * self.out_channels)
        else:
            # Average heads' outputs
            return aggr_out.mean(dim=1)